In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-07-03
Last modified on 2024-07-03
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, 
@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md
'''

'\nCreated on 2024-07-03\nLast modified on 2024-07-03\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, \n@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md\n'

**Índice de contenidos**

[Requerimientos]()

[Funciones]()

[Parámetros]()

[Ejecución principal]()


[1. Elementos generales]()

- [1.1. Generación de la matrix MITRE]()

- [1.2. Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada]()

- [1.3. Obtención de las TTP disponbles en Cyber Proof con regla de detección]()

[2. Tablas informativas]()

- [2.1. Técnicas]()

- [2.2. Tácticas]()

- [2.3. Data sources]()

- [2.4. Plataformas]()

- [2.6. Grupos]()

- [2.7. Software]()

[3. Relaciones]()

- [3.1. Relación técnicas - tácticas]()

- [3.2. Relación técnicas - data sources]()

- [3.3. Relación técnicas - plataformas]()

- [3.4. Relación técnicas - grupos]()

- [3.5. Relación grupos - software]()



[BONUS. GENERACIÓN LISTADO DE REGLAS DE DETECCIÓN POR TÉCNICA]()

#### **Requerimientos**

In [3]:
from stix2 import Filter, MemoryStore
import stix2
import requests

import os
import pandas as pd

import datetime
import csv

#### **Funciones**

In [4]:
def create_output_folder(path):
    '''
    Función encargada para crear el directorio facilitado en caso de no existir previamente.
    '''
    if not os.path.exists(path):
        os.makedirs(path)

In [5]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [6]:
def get_unique_ttps_generated(paths):
    '''
    Función encargada de retornar una lista de id de TTP's únicas generadas. Esta función recibirá una ruta generada previamente en la estructura de carpetas de guardado. 
    '''
    sub_folders = set()
    for path in paths:
        dir, file = os.path.split(path)
        last_sub_folder = os.path.basename(dir).strip()
        sub_folders.add(last_sub_folder)
    return list(sub_folders)

In [7]:
def compare_lists(list1, list2):
    '''
    Función para comparar el contenido de dos listas.
    '''
    set1 = set(list1)
    set2 = set(list2)
    # Items en comun
    common_elements = set1.intersection(set2)
    # Items unicamente en list1
    unique_in_list1 = set1.difference(set2)
    # Items unicamente en list2
    unique_in_list2 = set2.difference(set1)
    # Items en común
    num_common_elements = len(common_elements)
    # Items no coincidentes
    num_non_common_elements = len(unique_in_list1) + len(unique_in_list2)
    return num_common_elements, list(common_elements), list(unique_in_list1), list(unique_in_list2), num_non_common_elements

In [8]:
def unique_list(series):
    '''
    Función para devolver sólo items únicos al agregar
    '''
    return list(set(series))

In [9]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verificar si la solicitud fue exitosa
        stix_json = response.json()
        
        if "objects" in stix_json:
            return MemoryStore(stix_data=stix_json["objects"])
        else:
            raise ValueError("JSON no contiene la clave 'objects'")
            
    except requests.RequestException as e:
        print(f"Error al obtener datos de {matrix}-attack: {e}")
        return None

In [10]:
def save_df_as_csv(df, name, aux_folder='stix2', aux_folder_2=''):
    now = datetime.datetime.now()
    if not os.path.exists(os.path.join(os.getcwd(), 'outputs', aux_folder, aux_folder_2)):
        os.makedirs(os.path.join(os.getcwd(), 'outputs', aux_folder,aux_folder_2))
    file_to_save = os.path.join(os.getcwd(), 'outputs', aux_folder, aux_folder_2, (name+ '_'+ now.strftime('%d%m%Y_%H%M')+'h.csv'))
    df.to_csv(file_to_save, sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)
    print('Archivo guardado correctamente '+ name + '_' +now.strftime('%d%m%Y_%H%M')+'h.csv')

In [11]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [12]:
def get_CP_ttps_with_rules(mitre_matrix, way='file'):
    '''
    Función encargada de obtner el listado de TTP's para las que desde CP disponemos de reglas de detección. Para la ejecutarla correctamente debe de haberse ejecutado previamente el código del bloque get_rules_and_classify_by_ttp. Hay dos formas de evaluar las técnicas disponibles y son gestionadas por el parámetro "way". Por defecto el modo es "file", lo que nos indica que se va a evaluar uno de los ficheros generados de resumen de reglas por TTP clasificadas. En el caso de que el parámetro "way" recoja el valor "path", lo que hará es identificar las reglas meciante la exploración de subdirectorios.  
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix)
        
        if not os.path.exists(main_path):
            raise ValueError(f'No existe la path: {main_path}')
        
        if way == 'file':
            file_path = os.path.join(main_path, f'{mitre_matrix}-ttp_all_classified_rules.csv')  # Corrección aquí
            file = pd.read_csv(file_path, sep=';')
            file = file[file['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
            CP_techniques = file['ttp'].unique().tolist()  # Convertimos en lista de items únicos la columna que informa de la ttp
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)  # Ordenamos por longitud de caracteres para que cuando sea utilizada esta lista primero se evalúen las subtécnicas.

        elif way == 'path':
            CP_techniques = get_unique_ttps_generated(get_list_of_files_sub(main_path))
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)
            CP_techniques = [ttp for ttp in CP_techniques if ttp.startswith('T')]
            CP_techniques = [ttp for ttp in CP_techniques if ttp != 'T0000']

        else:
            CP_techniques = []
            
    except ValueError as e:
        print(f'{e}')
        CP_techniques = []
    
    return CP_techniques

In [13]:
def get_CP_and_NOCP_techniques_from_df(cp_techniques_list, df):
    '''
    Función encargada de filtrar un df dado el cual contiene el campo techniques_ID por las técnicas disponibles en CP (estas son facilitadas como parámetro en forma de lista). 
    '''
    cp_df = pd.DataFrame()
    nocp_df = pd.DataFrame() 
    try:
        cp_df = df[df['technique_ID'].isin(cp_techniques_list)]
        nocp_df = df[~df['technique_ID'].isin(cp_techniques_list)]
    except:
        print('No se ha podido ejecutar la operación.')
    return cp_df, nocp_df

In [14]:
def get_techniques_table(matrix_store, cp_techniques_list, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a la tecnicas (ID, nombre, url, descripción, estado deprecado, estado revocado).
    '''
    techniques_data = []
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    for tech in techniques:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in tech: 
            deprecated = tech['x_mitre_deprecated']
        if 'revoked' in tech: 
            revoked = tech['revoked']
        techniques_data.append({
            "technique_ID": tech['external_references'][0]['external_id'],
            "technique": tech['name'],
            "technique_url": tech['external_references'][0]['url'],
            "technique_description":tech['description'],
            "technique_deprecated": deprecated,
            "technique_revoked": revoked
        })
    MITRE_techniques_df = pd.DataFrame(techniques_data)
    if revoked_deprecated:
        MITRE_techniques_df = MITRE_techniques_df[(MITRE_techniques_df['technique_deprecated']!=True)&(MITRE_techniques_df['technique_revoked']!=True)]

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_df, NOCP_techniques_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_df)

    return MITRE_techniques_df, CP_techniques_df, NOCP_techniques_df

In [15]:
def get_tactics_table(matrix_store):
    '''
    Función que retorna el dataframe con la información relativa a la tácticas (ID, nombre, url, descripción, estado deprecado, estado revocado).
    '''
    tactics_data = []
    tactics = matrix_store.query([Filter('type', '=', 'x-mitre-tactic')])
    for tact in tactics:
        tactics_data.append({
            "tactic_ID": tact['external_references'][0]['external_id'],
            "tactic": tact['name'],
            "tactic_url": tact['external_references'][0]['url'],
            "tactic_description":tact['description']
        })
    MITRE_tactics_df = pd.DataFrame(tactics_data)
    return MITRE_tactics_df

In [16]:
def get_techniques_tactics_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y tácticas (N:N, 1:N y N:1), tambien devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    # Obtenemos las tácticas de la matriz
    tactics = matrix_store.query([
        Filter('type', '=', 'x-mitre-tactic')
    ])
    
    # Mediante diccionarios mapeamos la información que necesitamos
    technique_dict = {
        tech['external_references'][0]['external_id']: {
            'name': tech['name'],
            'deprecated': tech.get('x_mitre_deprecated', False),
            'revoked': tech.get('revoked', False)
        }
        for tech in techniques
        if 'external_references' in tech and len(tech['external_references']) > 0
    }
    tactic_dict = {
        tac['external_references'][0]['external_id']: tac['name']
        for tac in tactics
        if 'external_references' in tac and len(tac['external_references']) > 0
    }
    # Lista para almacenar las relaciones
    data = []

    # Recorrer todas las técnicas para encontrar sus relaciones con tácticas
    for technique in techniques:
        if 'kill_chain_phases' in technique:
            for phase in technique['kill_chain_phases']:
                # Cada fase representa una relación técnica-táctica
                tactic_shortname = phase['phase_name']
                external_id = technique['external_references'][0]['external_id'] if 'external_references' in technique and len(technique['external_references']) > 0 else None
                
                # Buscar el ID externo de la táctica correspondiente
                tactic_id = next((tac['external_references'][0]['external_id'] for tac in tactics if tac['x_mitre_shortname'] == tactic_shortname), None)
                # Si no hubiera ninguna relación
                if not external_id or not tactic_id:
                    continue
                # Agregamos la relación a la lista
                data.append({
                    "technique_ID": external_id,
                    "technique": technique_dict[external_id]['name'],
                    "tactic_ID": tactic_id,
                    "tactic": tactic_dict[tactic_id],
                    "technique_deprecated": technique_dict[external_id]['deprecated'],
                    "technique_revoked": technique_dict[external_id]['revoked']
                })
    
    # Creamos el dataframe final
    techniques_tactics_df = pd.DataFrame(data)
    #Filtramos técnicas deprecadas
    techniques_tactics_NN_df = techniques_tactics_df[(techniques_tactics_df['technique_deprecated']!=True)&(techniques_tactics_df['technique_revoked']!=True)]

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_tactics_NN_df = techniques_tactics_NN_df[['technique_ID', 'technique', 'tactic_ID', 'tactic']] # Filtramos las columnas que necesitamos
    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_tactics_NN_df)


    # Para finalizar agregamos por técnica y táctica con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y tácticas en una relación 1:N
    MITRE_technique_tactics_1N_df = MITRE_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    CP_technique_tactics_1N_df = CP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    NOCP_technique_tactics_1N_df = NOCP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })


    # Generamos el df compuesto por tecnicas y tácticas en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [17]:
def get_techniques_datasources_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y data sources (N:N, 1:N y N:1), tambien devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_ds_name = ''
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique: 
            ttp_ds_name = technique['x_mitre_data_sources']
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "data_source": ttp_ds_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)
    # Transformaciones de la tabla de relaciones de técnicas
    techniques_df = techniques_df.explode('data_source')
    techniques_df['data_source'] = techniques_df['data_source'].str.split(':').str[0] # El datasource al que aplica cada 
    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'data_source']]

    # Obtenemos los data sources de la matriz
    data_sources = matrix_store.query([
        Filter('type', '=', 'x-mitre-data-source')
    ])

    ds_data = []

    for data_source in data_sources:
        ds_id = ''
        ds_name = ''
        ds_revoked = ''
        ds_deprecated = ''
        if 'external_references' in data_source:
            ds_id = data_source['external_references'][0]['external_id']
        if 'name' in data_source:
            ds_name = data_source['name']
        if 'revoked' in data_source:
            ds_revoked = data_source['revoked']
        if 'x_mitre_deprecated' in data_source:
            ds_deprecated = data_source['x_mitre_deprecated']

        ds_data.append({
            "data_source_ID": ds_id,
            "data_source": ds_name,
            "data_source_deprecated": ds_deprecated,
            "data_source_revoked": ds_revoked
        })
    # Generamos df a partir de los datos recopilados
    data_sources_df = pd.DataFrame(ds_data)
    # Transformaciones de la tabla de relaciones de data sources
    data_sources_df = data_sources_df[(data_sources_df['data_source_deprecated']!=True)&(data_sources_df['data_source_revoked']!=True)]
    data_sources_df = data_sources_df[['data_source_ID', 'data_source']] 
    data_sources_df = data_sources_df.sort_values(by='data_source_ID')

    # Unimos las tablas
    techniques_data_sources_df = pd.merge(techniques_df, data_sources_df, on='data_source', how='left')
    techniques_data_sources_df = techniques_data_sources_df.drop_duplicates()

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_datasources_NN_df = techniques_data_sources_df[['technique_ID', 'technique', 'data_source_ID', 'data_source']]
    MITRE_techniques_datasources_NN_df = MITRE_techniques_datasources_NN_df.sort_values(by='technique_ID').reset_index(drop=True)
    MITRE_techniques_datasources_NN_df

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_datasources_NN_df)


    # Para finalizar agregamos por técnica y data sources con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y data sources en una relación 1:N
    MITRE_technique_datasources_1N_df = MITRE_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    CP_technique_datasources_1N_df = CP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    NOCP_technique_datasources_1N_df = NOCP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })

    # Generamos el df compuesto por tecnicas y data sources en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [18]:
def get_techniques_platforms_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y plataformas (N:N, 1:N y N:1), tambien devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_platform_name = ''
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique: 
            ttp_platform_name = technique['x_mitre_platforms']
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "platform": ttp_platform_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)

    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'platform']]
    techniques_df = techniques_df.explode('platform')
    MITRE_techniques_platforms_NN_df = techniques_df.drop_duplicates()
    MITRE_techniques_platforms_NN_df = MITRE_techniques_platforms_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_platforms_NN_df)


    # Para finalizar agregamos por técnica y plataformas con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y plataformas en una relación 1:N
    MITRE_technique_platforms_1N_df = MITRE_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    CP_technique_platforms_1N_df = CP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    NOCP_technique_platforms_1N_df = NOCP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_platform_N1_df = MITRE_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_platform_N1_df = CP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_platform_N1_df = NOCP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df

In [19]:
def get_techniques_groups_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y grupos (N:N, 1:N y N:1), tambien devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.
    '''
    # Verificamos que no haya habido algun error en la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Filtramos los objetos de tipo intrusion-set (grupos), attack-pattern (técnicas) y relationship (relaciones)
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])

    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # En 2 diccionarios mapeamos los IDs y nombres
    group_id_map = {g['id']: next((ref['external_id'] for ref in g['external_references'] if ref['source_name'] == 'mitre-attack'), None) for g in groups}
    technique_id_map = {t['id']: {
            'technique_id': next((ref['external_id'] for ref in t['external_references'] if ref['source_name'] == 'mitre-attack'), None),
            'technique_name': t['name']
        } for t in techniques}

    # Creamos un diccionario para mapear los grupos y las técnicas que usan mediante las relaciones
    group_techniques = []

    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'].startswith('intrusion-set') and rel['target_ref'].startswith('attack-pattern'):
            group_id = rel['source_ref']
            technique_id = rel['target_ref']

            group_abbrev_id = group_id_map.get(group_id)
            technique_info = technique_id_map.get(technique_id)
            group_name = next((g['name'] for g in groups if g['id'] == group_id), None)

            if group_abbrev_id and technique_info and group_name:
                group_techniques.append({
                    'group_ID': group_abbrev_id,
                    'group': group_name,
                    'technique_ID': technique_info['technique_id'],
                    'technique': technique_info['technique_name']
                })

    # Crear un dataframe con los resultados
    df = pd.DataFrame(group_techniques, columns=['group_ID', 'group', 'technique_ID', 'technique'])

    MITRE_techniques_groups_NN_df = df.drop_duplicates()
    MITRE_techniques_groups_NN_df = MITRE_techniques_groups_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_groups_NN_df)


    # Para finalizar agregamos por técnica y grupos con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y grupos en una relación 1:N
    MITRE_technique_groups_1N_df = MITRE_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })
    CP_technique_groups_1N_df = CP_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })
    NOCP_technique_groups_1N_df = NOCP_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_group_N1_df = MITRE_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_group_N1_df = CP_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_group_N1_df = NOCP_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_groups_NN_df, CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df, MITRE_technique_groups_1N_df, CP_technique_groups_1N_df, NOCP_technique_groups_1N_df, MITRE_techniques_group_N1_df, CP_techniques_group_N1_df, NOCP_techniques_group_N1_df

In [20]:
def get_groups_software_relationships(matrix_store):
    '''
    Función que retorna las relaciones entre grupos y software (N:N, 1:N y N:1). A diferencia de anteriores relaciones, en este caso no se puede filtrar por lista de técnicas disponibles. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.
    '''
    # Verificamos que no haya habido algun error en la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Filtramos los objetos de tipo intrusion-set (grupos), tool y malware (software) y relationship (relaciones)
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # Creamos diccionarios para mapear los IDs y nombres
    group_id_map = {g['id']: next((ref['external_id'] for ref in g['external_references'] if ref['source_name'] == 'mitre-attack'), None) for g in groups}
    software_id_map = {s['id']: {
            'software_ID': next((ref['external_id'] for ref in s['external_references'] if ref['source_name'] == 'mitre-attack'), None),
            'software': s['name']
        } for s in software}

    # Creamos un diccionario para mapear los grupos y el software que usan
    group_software = []

    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'].startswith('intrusion-set') and rel['target_ref'].startswith(('tool--', 'malware--')):
            group_id = rel['source_ref']
            software_id = rel['target_ref']

            group_abbrev_id = group_id_map.get(group_id)
            software_info = software_id_map.get(software_id)
            group_name = next((g['name'] for g in groups if g['id'] == group_id), None)

            if group_abbrev_id and software_info and group_name:
                group_software.append({
                    'group_ID': group_abbrev_id,
                    'group': group_name,
                    'software_ID': software_info['software_ID'],
                    'software': software_info['software']
                })

    # Dataframe con los resultados
    MITRE_groups_software_NN_df = pd.DataFrame(group_software, columns=['group_ID', 'group', 'software_ID', 'software'])

    # Para finalizar agregamos por grupo y software con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por grupo y software's en una relación 1:N
    MITRE_group_software_1N_df = MITRE_groups_software_NN_df.groupby('group_ID', as_index=False).agg({
    'group':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })

    # Generamos el df compuesto por grupos y software en una relación N:1
    MITRE_groups_software_N1_df = MITRE_groups_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_groups_software_NN_df, MITRE_group_software_1N_df, MITRE_groups_software_N1_df


### **Parámetros**

In [21]:
matrix = 'enterprise'
save_as_csv = True
debug_df = True # Parámetro para controlar el printeado de df

# **Ejecución principal**

## **1. Elementos generales**

### **1.1 Generación de la matriz MITRE**

In [22]:
mitre_matrix = get_data_from_branch(matrix)
mitre_matrix

### **1.2 Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada**

In [23]:
techniques_list = get_list_techniques_from_stix2(mitre_matrix,'both')
print(f"Se han generado la lista de técnicas (ID) para la matriz {matrix.upper()} que contiene un total de {len(techniques_list)} TTP's")

Se han generado la lista de técnicas (ID) para la matriz ENTERPRISE que contiene un total de 637 TTP's


### **1.3 Obtención de las TTP disponbles en Cyber Proof con regla de detección**

In [24]:
cp_techniques =  get_CP_ttps_with_rules(matrix, way='file')
print(f"Se han obtenido un total de {len(cp_techniques)} TTP's con regla de detección asociada disponibles en CP.")

Se han obtenido un total de 521 TTP's con regla de detección asociada disponibles en CP.


## **2. Tablas informativas**

### **2.1. Técnicas**

#### **Generación de las tablas**

In [25]:
MITRE_techniques_df, CP_techniques_df, NOCP_techniques_df = get_techniques_table(mitre_matrix, cp_techniques, revoked_deprecated=True)

#### **Debug**

In [26]:
if debug_df:
    display(MITRE_techniques_df.head(3))
    display(CP_techniques_df.head(3))
    display(NOCP_techniques_df.head(3))
    print(f'{MITRE_techniques_df.shape}, {CP_techniques_df.shape}, {NOCP_techniques_df.shape}')

,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked
0,T1055.011,Extra Window Memory Injection,https://attack.mitre.org/techniques/T1055/011,Adversaries may inject malicious code into pro...,False,False
1,T1053.005,Scheduled Task,https://attack.mitre.org/techniques/T1053/005,Adversaries may abuse the Windows Task Schedul...,False,False
2,T1205.002,Socket Filters,https://attack.mitre.org/techniques/T1205/002,Adversaries may attach filters to a network so...,False,False


,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked
0,T1055.011,Extra Window Memory Injection,https://attack.mitre.org/techniques/T1055/011,Adversaries may inject malicious code into pro...,False,False
1,T1053.005,Scheduled Task,https://attack.mitre.org/techniques/T1053/005,Adversaries may abuse the Windows Task Schedul...,False,False
4,T1560.001,Archive via Utility,https://attack.mitre.org/techniques/T1560/001,Adversaries may use utilities to compress and/...,False,False


,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked
2,T1205.002,Socket Filters,https://attack.mitre.org/techniques/T1205/002,Adversaries may attach filters to a network so...,False,False
9,T1027.011,Fileless Storage,https://attack.mitre.org/techniques/T1027/011,"Adversaries may store data in ""fileless"" forma...",False,False
17,T1583.007,Serverless,https://attack.mitre.org/techniques/T1583/007,Adversaries may purchase and configure serverl...,False,False


(637, 6), (521, 6), (116, 6)


#### **Guardado**

In [27]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_df, f'[MITRE]_{matrix}_techniques','stix2', 'information/Techniques')
    save_df_as_csv(CP_techniques_df, f'[CP]_{matrix}_techniques','stix2', 'information/Techniques')
    save_df_as_csv(NOCP_techniques_df, f'[NOCP]_{matrix}_techniques','stix2', 'information/Techniques')

Archivo guardado correctamente [MITRE]_enterprise_techniques_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_08072024_1401h.csv


### **2.2. Tácticas**

#### **Generación de la tabla**

In [28]:
MITRE_tactics_df = get_tactics_table(mitre_matrix)

#### **Debug**

In [29]:
if debug_df:
    display(MITRE_tactics_df.head(3))
    print(f'{MITRE_tactics_df.shape}')

,tactic_ID,tactic,tactic_url,tactic_description
0,TA0006,Credential Access,https://attack.mitre.org/tactics/TA0006,The adversary is trying to steal account names...
1,TA0002,Execution,https://attack.mitre.org/tactics/TA0002,The adversary is trying to run malicious code....
2,TA0040,Impact,https://attack.mitre.org/tactics/TA0040,"The adversary is trying to manipulate, interru..."


(14, 4)


#### **Guardado**

In [30]:
if save_as_csv:
    save_df_as_csv(MITRE_tactics_df, f'[MITRE]_{matrix}_tactics','stix2', 'information/Tactics')

Archivo guardado correctamente [MITRE]_enterprise_tactics_08072024_1401h.csv


In [31]:
# tactics_data = []
# tactics = mitre_matrix.query([Filter('type', '=', 'x-mitre-tactic')])
# for tact in tactics:
#     deprecated = False
#     revoked = False
#     if 'x_mitre_deprecated' in tact: 
#         deprecated = tact['x_mitre_deprecated']
#     if 'revoked' in tact: 
#         revoked = tact['revoked']

#     tactics_data.append({
#         "tactic_ID": tact['external_references'][0]['external_id'],
#         "tactic": tact['name'],
#         "tactic_url": tact['external_references'][0]['url'],
#         "tactic_description":tact['description'],
#         "tactic_deprecated": deprecated,
#         "tactic_revoked": revoked
#     })

# MITRE_tactics_df = pd.DataFrame(tactics_data)
# MITRE_tactics_df

#### **Generación de las tablas**

#### **Debug**

#### **Guardado**

## **3. Relaciones**

### **3.1. Relación técnicas - tácticas**

#### **Generación de las tablas**

In [32]:
MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df = get_techniques_tactics_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [33]:
if debug_df:
    display(MITRE_techniques_tactics_NN_df.head(3))
    display(CP_techniques_tactics_NN_df.head(3))
    display(NOCP_techniques_tactics_NN_df.head(3))
    print(f'{MITRE_techniques_tactics_NN_df.shape}, {CP_techniques_tactics_NN_df.shape}, {NOCP_techniques_tactics_NN_df.shape}')

,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation
2,T1053.005,Scheduled Task,TA0002,Execution


,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation
2,T1053.005,Scheduled Task,TA0002,Execution


,technique_ID,technique,tactic_ID,tactic
5,T1205.002,Socket Filters,TA0005,Defense Evasion
6,T1205.002,Socket Filters,TA0003,Persistence
7,T1205.002,Socket Filters,TA0011,Command and Control


(830, 4), (686, 4), (144, 4)


In [34]:
if debug_df:
    display(MITRE_technique_tactics_1N_df.head(3))
    display(CP_technique_tactics_1N_df.head(3))
    display(NOCP_technique_tactics_1N_df.head(3))
    print(f'{MITRE_technique_tactics_1N_df.shape}, {CP_technique_tactics_1N_df.shape}, {NOCP_technique_tactics_1N_df.shape}')

,technique_ID,technique,tactic_ID,tactic
0,T1001,[Data Obfuscation],[TA0011],[Command and Control]
1,T1001.001,[Junk Data],[TA0011],[Command and Control]
2,T1001.002,[Steganography],[TA0011],[Command and Control]


,technique_ID,technique,tactic_ID,tactic
0,T1001,[Data Obfuscation],[TA0011],[Command and Control]
1,T1001.002,[Steganography],[TA0011],[Command and Control]
2,T1001.003,[Protocol Impersonation],[TA0011],[Command and Control]


,technique_ID,technique,tactic_ID,tactic
0,T1001.001,[Junk Data],[TA0011],[Command and Control]
1,T1011.001,[Exfiltration Over Bluetooth],[TA0010],[Exfiltration]
2,T1016.002,[Wi-Fi Discovery],[TA0007],[Discovery]


(637, 4), (521, 4), (116, 4)


In [35]:
if debug_df:
    display(MITRE_techniques_tactic_N1_df.head(3))
    display(CP_techniques_tactic_N1_df.head(3))
    display(NOCP_techniques_tactic_N1_df.head(3))
    print(f'{MITRE_techniques_tactic_N1_df.shape}, {CP_techniques_tactic_N1_df.shape}, {NOCP_techniques_tactic_N1_df.shape}')

,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1195.002, T1199, T1078.001, T1566.004, T1189...","[Replication Through Removable Media, Content ..."
1,TA0002,[Execution],"[T1059.001, T1204.001, T1609, T1053, T1106, T1...","[Deploy Container, Cloud API, Unix Shell, Soft..."
2,TA0003,[Persistence],"[T1547.009, T1546.008, T1098.003, T1053.007, T...","[Path Interception by Unquoted Path, BITS Jobs..."


,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1091, T1566.001, T1078, T1078.003, T1566, T1...","[Trusted Relationship, Exploit Public-Facing A..."
1,TA0002,[Execution],"[T1059.001, T1204.001, T1609, T1053, T1106, T1...","[Deploy Container, Cloud API, Unix Shell, Soft..."
2,TA0003,[Persistence],"[T1547.009, T1546.008, T1098.003, T1053.007, T...","[Path Interception by Unquoted Path, BITS Jobs..."


,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1659, T1195.003, T1566.004]","[Content Injection, Spearphishing Voice, Compr..."
1,TA0002,[Execution],"[T1059.010, T1559.003, T1651, T1059.008]","[AutoHotKey & AutoIT, Network Device CLI, XPC ..."
2,TA0003,[Persistence],"[T1137.004, T1205.002, T1574.004, T1556.005, T...","[AppDomainManager, Login Hook, Reversible Encr..."


(14, 4), (14, 4), (13, 4)


#### **Guardado**

In [36]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_tactics_NN_df, f'[MITRE]_{matrix}_techniques_tactics_NN','stix2', 'relations/Techniques_Tactics')
    save_df_as_csv(CP_techniques_tactics_NN_df, f'[CP]_{matrix}_techniques_tactics_NN','stix2', 'relations/Techniques_Tactics')
    save_df_as_csv(NOCP_techniques_tactics_NN_df, f'[NOCP]_{matrix}_techniques_tactics_NN','stix2', 'relations/Techniques_Tactics')

    save_df_as_csv(MITRE_technique_tactics_1N_df, f'[MITRE]_{matrix}_technique_tactics_1N','stix2', 'relations/Techniques_Tactics')
    save_df_as_csv(CP_technique_tactics_1N_df, f'[CP]_{matrix}_technique_tactics_1N','stix2', 'relations/Techniques_Tactics')
    save_df_as_csv(NOCP_technique_tactics_1N_df, f'[NOCP]_{matrix}_technique_tactics_1N','stix2', 'relations/Techniques_Tactics')

    save_df_as_csv(MITRE_techniques_tactic_N1_df, f'[MITRE]_{matrix}_techniques_tactic_N1','stix2', 'relations/Techniques_Tactics')
    save_df_as_csv(CP_techniques_tactic_N1_df, f'[CP]_{matrix}_techniques_tactic_N1','stix2', 'relations/Techniques_Tactics')
    save_df_as_csv(NOCP_techniques_tactic_N1_df, f'[NOCP]_{matrix}_techniques_tactic_N1','stix2', 'relations/Techniques_Tactics')

Archivo guardado correctamente [MITRE]_enterprise_techniques_tactics_NN_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_tactics_NN_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_tactics_NN_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_tactics_1N_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_technique_tactics_1N_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_tactics_1N_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_tactic_N1_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_tactic_N1_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_tactic_N1_08072024_1401h.csv


### **3.2. Relación técnicas - data sources**

#### **Generación de las tablas**

In [37]:
MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_datasource_N1_df, CP_techniques_datasource_N1_df, NOCP_techniques_datasource_N1_df = get_techniques_datasources_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [38]:
if debug_df:
    display(MITRE_techniques_datasources_NN_df.head(3))
    display(CP_techniques_datasources_NN_df.head(3))
    display(NOCP_techniques_datasources_NN_df.head(3))
    print(f'{MITRE_techniques_datasources_NN_df.shape}, {CP_techniques_datasources_NN_df.shape}, {NOCP_techniques_datasources_NN_df.shape}')

,technique_ID,technique,data_source_ID,data_source
0,T1001,Data Obfuscation,DS0029,Network Traffic
1,T1001.001,Junk Data,DS0029,Network Traffic
2,T1001.002,Steganography,DS0029,Network Traffic


,technique_ID,technique,data_source_ID,data_source
0,T1001,Data Obfuscation,DS0029,Network Traffic
2,T1001.002,Steganography,DS0029,Network Traffic
3,T1001.003,Protocol Impersonation,DS0029,Network Traffic


,technique_ID,technique,data_source_ID,data_source
1,T1001.001,Junk Data,DS0029,Network Traffic
44,T1011.001,Exfiltration Over Bluetooth,DS0029,Network Traffic
45,T1011.001,Exfiltration Over Bluetooth,DS0017,Command


(1594, 4), (1401, 4), (193, 4)


In [39]:
if debug_df:
    display(MITRE_technique_datasources_1N_df.head(3))
    display(CP_technique_datasources_1N_df.head(3))
    display(NOCP_technique_datasources_1N_df.head(3))
    print(f'{MITRE_technique_datasources_1N_df.shape}, {CP_technique_datasources_1N_df.shape}, {NOCP_technique_datasources_1N_df.shape}')

,technique_ID,technique,data_source_ID,data_source
0,T1001,[Data Obfuscation],[DS0029],[Network Traffic]
1,T1001.001,[Junk Data],[DS0029],[Network Traffic]
2,T1001.002,[Steganography],[DS0029],[Network Traffic]


,technique_ID,technique,data_source_ID,data_source
0,T1001,[Data Obfuscation],[DS0029],[Network Traffic]
1,T1001.002,[Steganography],[DS0029],[Network Traffic]
2,T1001.003,[Protocol Impersonation],[DS0029],[Network Traffic]


,technique_ID,technique,data_source_ID,data_source
0,T1001.001,[Junk Data],[DS0029],[Network Traffic]
1,T1011.001,[Exfiltration Over Bluetooth],"[DS0029, DS0022, DS0017]","[File, Command, Network Traffic]"
2,T1016.002,[Wi-Fi Discovery],"[DS0009, DS0017]","[Process, Command]"


(637, 4), (521, 4), (116, 4)


In [40]:
if debug_df:
    display(MITRE_techniques_datasource_N1_df.head(3))
    display(CP_techniques_datasource_N1_df.head(3))
    display(NOCP_techniques_datasource_N1_df.head(3))
    print(f'{MITRE_techniques_datasource_N1_df.shape}, {CP_techniques_datasource_N1_df.shape}, {NOCP_techniques_datasource_N1_df.shape}')

,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T1542.004, T1542.001, T1564, T1542, T1542.005...","[Rootkit, TFTP Boot, Hidden File System, ROMMO..."
1,DS0002,[User Account],"[T1550.003, T1212, T1207, T1538, T1098.003, T1...","[Password Guessing, Create Account, Indicator ..."
2,DS0003,[Scheduled Job],"[T1053.007, T1036, T1053, T1036.004, T1053.002...","[Masquerading, Scheduled Task, Indicator Remov..."


,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T1542.001, T1564, T1542, T1542.005, T1564.005...","[Rootkit, TFTP Boot, Hidden File System, Syste..."
1,DS0002,[User Account],"[T1550.003, T1212, T1207, T1538, T1098.003, T1...","[Password Guessing, Create Account, Indicator ..."
2,DS0003,[Scheduled Job],"[T1053.007, T1036, T1053, T1036.004, T1053.002...","[Masquerading, Scheduled Task, Indicator Remov..."


,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T1542.004, T1542.002]","[ROMMONkit, Component Firmware]"
1,DS0002,[User Account],"[T1548.005, T1098.006, T1098.005, T1556.005]","[Additional Container Cluster Roles, Temporary..."
2,DS0004,[Malware Repository],[T1587.002],[Code Signing Certificates]


(37, 4), (37, 4), (23, 4)


#### **Guardado**

In [41]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_datasources_NN_df, f'[MITRE]_{matrix}_techniques_datasources_NN','stix2', 'relations/Techniques_DataSources')
    save_df_as_csv(CP_techniques_datasources_NN_df, f'[CP]_{matrix}_techniques_datasources_NN','stix2', 'relations/Techniques_DataSources')
    save_df_as_csv(NOCP_techniques_datasources_NN_df, f'[NOCP]_{matrix}_techniques_datasources_NN','stix2', 'relations/Techniques_DataSources')

    save_df_as_csv(MITRE_technique_datasources_1N_df, f'[MITRE]_{matrix}_technique_datasources_1N','stix2', 'relations/Techniques_DataSources')
    save_df_as_csv(CP_technique_datasources_1N_df, f'[CP]_{matrix}_technique_datasources_1N','stix2', 'relations/Techniques_DataSources')
    save_df_as_csv(NOCP_technique_datasources_1N_df, f'[NOCP]_{matrix}_technique_datasources_1N','stix2', 'relations/Techniques_DataSources')

    save_df_as_csv(MITRE_techniques_datasource_N1_df, f'[MITRE]_{matrix}_techniques_datasource_N1','stix2', 'relations/Techniques_DataSources')
    save_df_as_csv(CP_techniques_datasource_N1_df, f'[CP]_{matrix}_techniques_datasource_N1','stix2', 'relations/Techniques_DataSources')
    save_df_as_csv(NOCP_techniques_datasource_N1_df, f'[NOCP]_{matrix}_techniques_datasource_N1','stix2', 'relations/Techniques_DataSources')

Archivo guardado correctamente [MITRE]_enterprise_techniques_datasources_NN_08072024_1401h.csv


Archivo guardado correctamente [CP]_enterprise_techniques_datasources_NN_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_datasources_NN_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_datasources_1N_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_technique_datasources_1N_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_datasources_1N_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_datasource_N1_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_datasource_N1_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_datasource_N1_08072024_1401h.csv


### **3.3. Relación técnicas - plataformas**

#### **Generación de las tablas**

In [42]:
MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df = get_techniques_platforms_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [43]:
if debug_df:
    display(MITRE_techniques_platforms_NN_df.head(3))
    display(CP_techniques_platforms_NN_df.head(3))
    display(NOCP_techniques_platforms_NN_df.head(3))
    print(f'{MITRE_techniques_platforms_NN_df.shape}, {CP_techniques_platforms_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

,technique_ID,technique,platform
0,T1001,Data Obfuscation,Linux
1,T1001,Data Obfuscation,macOS
2,T1001,Data Obfuscation,Windows


,technique_ID,technique,platform
0,T1001,Data Obfuscation,Linux
1,T1001,Data Obfuscation,macOS
2,T1001,Data Obfuscation,Windows


,technique_ID,technique,platform
3,T1001.001,Junk Data,Windows
4,T1001.001,Junk Data,Linux
5,T1001.001,Junk Data,macOS


(1654, 3), (1436, 3), (218, 3)


In [44]:
if debug_df:
    display(MITRE_technique_platforms_1N_df.head(3))
    display(CP_technique_platforms_1N_df.head(3))
    display(NOCP_technique_platforms_1N_df.head(3))
    print(f'{MITRE_technique_platforms_1N_df.shape}, {CP_technique_platforms_1N_df.shape}, {NOCP_technique_platforms_1N_df.shape}')

,technique_ID,technique,platform
0,T1001,[Data Obfuscation],"[Windows, Linux, macOS]"
1,T1001.001,[Junk Data],"[Windows, Linux, macOS]"
2,T1001.002,[Steganography],"[Windows, Linux, macOS]"


,technique_ID,technique,platform
0,T1001,[Data Obfuscation],"[Windows, Linux, macOS]"
1,T1001.002,[Steganography],"[Windows, Linux, macOS]"
2,T1001.003,[Protocol Impersonation],"[Windows, Linux, macOS]"


,technique_ID,technique,platform
0,T1001.001,[Junk Data],"[Windows, Linux, macOS]"
1,T1011.001,[Exfiltration Over Bluetooth],"[Windows, Linux, macOS]"
2,T1016.002,[Wi-Fi Discovery],"[Windows, Linux, macOS]"


(637, 3), (521, 3), (116, 3)


In [45]:
if debug_df:
    display(MITRE_techniques_platform_N1_df.head(3))
    display(CP_techniques_platform_N1_df.head(3))
    display(NOCP_techniques_platform_N1_df.head(3))
    print(f'{MITRE_techniques_platform_N1_df.shape}, {CP_techniques_platform_N1_df.shape}, {NOCP_techniques_platform_N1_df.shape}')

,platform,technique_ID,technique
0,,"[T1588.005, T1596.004, T1596.005, T1591.001, T...","[Network Topology, Scan Databases, Botnet, Net..."
1,Azure AD,"[T1069, T1212, T1484.002, T1069.003, T1538, T1...","[Password Guessing, Create Account, Cloud API,..."
2,Containers,"[T1069, T1609, T1053, T1562, T1078.001, T1110,...","[Password Guessing, Create Account, Deploy Con..."


,platform,technique_ID,technique
0,,"[T1591.001, T1591, T1586.003, T1596, T1590.002...","[Network Topology, Network Trust Dependencies,..."
1,Azure AD,"[T1069, T1212, T1484.002, T1069.003, T1538, T1...","[Password Guessing, Create Account, Cloud API,..."
2,Containers,"[T1069, T1609, T1053, T1562, T1078.001, T1110,...","[Password Guessing, Create Account, Deploy Con..."


,platform,technique_ID,technique
0,,"[T1593.001, T1588.005, T1596.003, T1596.001, T...","[CDNs, Scan Databases, Botnet, DNS/Passive DNS..."
1,Azure AD,"[T1548.005, T1098.005, T1556.009]","[Temporary Elevated Cloud Access, Conditional ..."
2,Containers,"[T1098.006, T1543.005]","[Additional Container Cluster Roles, Container..."


(12, 3), (12, 3), (12, 3)


#### **Guardado**

In [46]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_platforms_NN_df, f'[MITRE]_{matrix}_techniques_platforms_NN','stix2', 'relations/Techniques_Platforms')
    save_df_as_csv(CP_techniques_platforms_NN_df, f'[CP]_{matrix}_techniques_platforms_NN','stix2', 'relations/Techniques_Platforms')
    save_df_as_csv(NOCP_techniques_platforms_NN_df, f'[NOCP]_{matrix}_techniques_platforms_NN','stix2', 'relations/Techniques_Platforms')

    save_df_as_csv(MITRE_technique_platforms_1N_df, f'[MITRE]_{matrix}_technique_platforms_1N','stix2', 'relations/Techniques_Platforms')
    save_df_as_csv(CP_technique_platforms_1N_df, f'[CP]_{matrix}_technique_platforms_1N','stix2', 'relations/Techniques_Platforms')
    save_df_as_csv(NOCP_technique_platforms_1N_df, f'[NOCP]_{matrix}_technique_platforms_1N','stix2', 'relations/Techniques_Platforms')

    save_df_as_csv(MITRE_techniques_platform_N1_df, f'[MITRE]_{matrix}_techniques_platform_N1','stix2', 'relations/Techniques_Platforms')
    save_df_as_csv(CP_techniques_platform_N1_df, f'[CP]_{matrix}_techniques_platform_N1','stix2', 'relations/Techniques_Platforms')
    save_df_as_csv(NOCP_techniques_platform_N1_df, f'[NOCP]_{matrix}_techniques_platform_N1','stix2', 'relations/Techniques_Platforms')

Archivo guardado correctamente [MITRE]_enterprise_techniques_platforms_NN_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_platforms_NN_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_platforms_NN_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_platforms_1N_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_technique_platforms_1N_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_platforms_1N_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_platform_N1_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_platform_N1_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_platform_N1_08072024_1401h.csv


### **3.4. Relación técnicas - grupos**

#### **Generación de las tablas**

In [47]:
MITRE_techniques_groups_NN_df, CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df, MITRE_technique_groups_1N_df, CP_technique_groups_1N_df, NOCP_technique_groups_1N_df, MITRE_techniques_group_N1_df, CP_techniques_group_N1_df, NOCP_techniques_group_N1_df = get_techniques_groups_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [48]:
if debug_df:
    display(MITRE_techniques_groups_NN_df.head(3))
    display(CP_techniques_groups_NN_df.head(3))
    display(NOCP_techniques_groups_NN_df.head(3))
    print(f'{MITRE_techniques_groups_NN_df.shape}, {CP_techniques_groups_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

,group_ID,group,technique_ID,technique
0,G0007,APT28,T1001.001,Junk Data
1,G0001,Axiom,T1001.002,Steganography
2,G0126,Higaisa,T1001.003,Protocol Impersonation


,group_ID,group,technique_ID,technique
1,G0001,Axiom,T1001.002,Steganography
2,G0126,Higaisa,T1001.003,Protocol Impersonation
3,G0032,Lazarus Group,T1001.003,Protocol Impersonation


,group_ID,group,technique_ID,technique
0,G0007,APT28,T1001.001,Junk Data
209,G0059,Magic Hound,T1016.002,Wi-Fi Discovery
316,G0007,APT28,T1025,Data from Removable Media


(3387, 4), (3163, 4), (218, 3)


In [49]:
if debug_df:
    display(MITRE_technique_groups_1N_df.head(3))
    display(CP_technique_groups_1N_df.head(3))
    display(NOCP_technique_groups_1N_df.head(3))
    print(f'{MITRE_technique_groups_1N_df.shape}, {CP_technique_groups_1N_df.shape}, {NOCP_technique_groups_1N_df.shape}')

,technique_ID,technique,group_ID,group
0,T1001.001,[Junk Data],[G0007],[APT28]
1,T1001.002,[Steganography],[G0001],[Axiom]
2,T1001.003,[Protocol Impersonation],"[G0032, G0126]","[Lazarus Group, Higaisa]"


,technique_ID,technique,group_ID,group
0,T1001.002,[Steganography],[G0001],[Axiom]
1,T1001.003,[Protocol Impersonation],"[G0032, G0126]","[Lazarus Group, Higaisa]"
2,T1003,[OS Credential Dumping],"[G0087, G0039, G0131, G0054, G0065, G0007, G00...","[APT39, Leviathan, Tonto Team, Poseidon Group,..."


,technique_ID,technique,group_ID,group
0,T1001.001,[Junk Data],[G0007],[APT28]
1,T1016.002,[Wi-Fi Discovery],[G0059],[Magic Hound]
2,T1025,[Data from Removable Media],"[G0007, G0047, G0010]","[Turla, APT28, Gamaredon Group]"


(421, 4), (365, 4), (56, 4)


In [50]:
if debug_df:
    display(MITRE_techniques_group_N1_df.head(3))
    display(CP_techniques_group_N1_df.head(3))
    display(NOCP_techniques_group_N1_df.head(3))
    print(f'{MITRE_techniques_group_N1_df.shape}, {CP_techniques_group_N1_df.shape}, {NOCP_techniques_group_N1_df.shape}')

,group_ID,group,technique_ID,technique
0,G0001,[Axiom],"[T1553, T1560, T1546.008, T1003, T1078, T1001....","[OS Credential Dumping, Exploit Public-Facing ..."
1,G0002,[Moafee],[T1027.001],[Binary Padding]
2,G0003,[Cleaver],"[T1003.001, T1585.001, T1588.002, T1587.001, T...","[Social Media Accounts, LSASS Memory, Malware,..."


,group_ID,group,technique_ID,technique
0,G0001,[Axiom],"[T1553, T1560, T1546.008, T1003, T1078, T1001....","[OS Credential Dumping, Exploit Public-Facing ..."
1,G0002,[Moafee],[T1027.001],[Binary Padding]
2,G0003,[Cleaver],"[T1003.001, T1588.002, T1587.001, T1557.002]","[ARP Cache Poisoning, Tool, LSASS Memory, Malw..."


,group_ID,group,technique_ID,technique
0,G0001,[Axiom],"[T1583.003, T1584.005, T1583.002]","[DNS Server, Virtual Private Server, Botnet]"
1,G0003,[Cleaver],[T1585.001],[Social Media Accounts]
2,G0005,[APT12],[T1568.003],[DNS Calculation]


(143, 4), (142, 4), (81, 4)


#### **Guardado**

In [51]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_groups_NN_df, f'[MITRE]_{matrix}_techniques_groups_NN','stix2', 'relations/Techniques_Groups')
    save_df_as_csv(CP_techniques_groups_NN_df, f'[CP]_{matrix}_techniques_groups_NN','stix2', 'relations/Techniques_Groups')
    save_df_as_csv(NOCP_techniques_groups_NN_df, f'[NOCP]_{matrix}_techniques_groups_NN','stix2', 'relations/Techniques_Groups')

    save_df_as_csv(MITRE_technique_groups_1N_df, f'[MITRE]_{matrix}_technique_groups_1N','stix2', 'relations/Techniques_Groups')
    save_df_as_csv(CP_technique_groups_1N_df, f'[CP]_{matrix}_technique_groups_1N','stix2', 'relations/Techniques_Groups')
    save_df_as_csv(NOCP_technique_groups_1N_df, f'[NOCP]_{matrix}_technique_groups_1N','stix2', 'relations/Techniques_Groups')

    save_df_as_csv(MITRE_techniques_group_N1_df, f'[MITRE]_{matrix}_techniques_group_N1','stix2', 'relations/Techniques_Groups')
    save_df_as_csv(CP_techniques_group_N1_df, f'[CP]_{matrix}_techniques_group_N1','stix2', 'relations/Techniques_Groups')
    save_df_as_csv(NOCP_techniques_group_N1_df, f'[NOCP]_{matrix}_techniques_group_N1','stix2', 'relations/Techniques_Groups')

Archivo guardado correctamente [MITRE]_enterprise_techniques_groups_NN_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_groups_NN_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_groups_NN_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_groups_1N_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_technique_groups_1N_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_groups_1N_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_group_N1_08072024_1401h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_group_N1_08072024_1401h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_group_N1_08072024_1401h.csv


### **3.5. Relación grupos - software**

#### **Generación de las tablas**

In [52]:
MITRE_groups_software_NN_df, MITRE_group_software_1N_df, MITRE_groups_software_N1_df = get_groups_software_relationships(mitre_matrix)

Dataframes generados correctamente!


#### **Debug**

In [53]:
if debug_df:
    display(MITRE_groups_software_NN_df.head(3))
    display(MITRE_group_software_1N_df.head(3))
    display(MITRE_groups_software_N1_df.head(3))
    print(f'{MITRE_groups_software_NN_df.shape}, {MITRE_group_software_1N_df.shape}, {MITRE_groups_software_N1_df.shape}')

,group_ID,group,software_ID,software
0,G0066,Elderwood,S0012,PoisonIvy
1,G0049,OilRig,S0189,ISMInjector
2,G0032,Lazarus Group,S0364,RawDisk


,group_ID,group,software_ID,software
0,G0001,[Axiom],"[S0672, S0009, S0021, S0203, S0013, S0412, S00...","[Hydraq, PlugX, ZxShell, gh0st RAT, Zox, Hikit..."
1,G0002,[Moafee],[S0012],[PoisonIvy]
2,G0003,[Cleaver],"[S0004, S0029, S0002, S0056]","[Mimikatz, PsExec, TinyZBot, Net Crawler]"


,software_ID,software,group_ID,group
0,S0002,[Mimikatz],"[G0016, G0049, G0037, G0007, G0003, G0046, G10...","[Cobalt Group, Blue Mockingbird, Indrik Spider..."
1,S0003,[RIPTIDE],[G0005],[APT12]
2,S0004,[TinyZBot],[G0003],[Cleaver]


(921, 4), (136, 4), (498, 4)


#### **Guardado**

In [54]:
if save_as_csv:
    save_df_as_csv(MITRE_groups_software_NN_df, f'[MITRE]_{matrix}_groups_software_NN','stix2', 'relations/Groups_Software')
    save_df_as_csv(MITRE_group_software_1N_df, f'[MITRE]_{matrix}_group_software_1N','stix2', 'relations/Groups_Software')
    save_df_as_csv(MITRE_groups_software_N1_df, f'[MITRE]_{matrix}_groups_software_N1','stix2', 'relations/Groups_Software')

Archivo guardado correctamente [MITRE]_enterprise_groups_software_NN_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_group_software_1N_08072024_1401h.csv
Archivo guardado correctamente [MITRE]_enterprise_groups_software_N1_08072024_1401h.csv


In [55]:
1/0

ZeroDivisionError: division by zero

**TESTS**

In [ ]:
mitre_matrix.query([Filter("id", "=", "attack-pattern--65f2d882-3f41-4d48-8a06-29af77ec9f90")])[0]

AttackPattern(type='attack-pattern', id='attack-pattern--65f2d882-3f41-4d48-8a06-29af77ec9f90', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2020-02-11T18:41:44.783Z', modified='2023-12-27T17:57:20.003Z', name='LSASS Memory', description="Adversaries may attempt to access credential material stored in the process memory of the Local Security Authority Subsystem Service (LSASS). After a user logs on, the system generates and stores a variety of credential materials in LSASS process memory. These credential materials can be harvested by an administrative user or SYSTEM and used to conduct [Lateral Movement](https://attack.mitre.org/tactics/TA0008) using [Use Alternate Authentication Material](https://attack.mitre.org/techniques/T1550).\n\nAs well as in-memory techniques, the LSASS process memory can be dumped from the target host and analyzed on a local system.\n\nFor example, on the target host use procdump:\n\n* <code>procdump -ma lsass.exe lsass_dump</code

**Intentando acceder segun documentación**

In [ ]:
def remove_revoked_deprecated(stix_objects):
    """Remove any revoked or deprecated objects from queries made to the data source"""
    # Note we use .get() because the property may not be present in the JSON data. The default is False
    # if the property is not set.
    return list(
        filter(
            lambda x: x.get("x_mitre_deprecated", False) is False and x.get("revoked", False) is False,
            stix_objects
        )
    )
def get_related(thesrc, src_type, rel_type, target_type, reverse=False):
    """build relationship mappings
       params:
         thesrc: MemoryStore to build relationship lookups for
         src_type: source type for the relationships, e.g "attack-pattern"
         rel_type: relationship type for the relationships, e.g "uses"
         target_type: target type for the relationship, e.g "intrusion-set"
         reverse: build reverse mapping of target to source
    """

    relationships = thesrc.query([
        Filter('type', '=', 'relationship'),
        Filter('relationship_type', '=', rel_type),
        Filter('revoked', '=', False),
    ])

    # See section below on "Removing revoked and deprecated objects"
    relationships = remove_revoked_deprecated(relationships)

    # stix_id => [ { relationship, related_object_id } for each related object ]
    id_to_related = {}

    # build the dict
    for relationship in relationships:
        if src_type in relationship.source_ref and target_type in relationship.target_ref:
            if (relationship.source_ref in id_to_related and not reverse) or (relationship.target_ref in id_to_related and reverse):
                # append to existing entry
                if not reverse:
                    id_to_related[relationship.source_ref].append({
                        "relationship": relationship,
                        "id": relationship.target_ref
                    })
                else:
                    id_to_related[relationship.target_ref].append({
                        "relationship": relationship,
                        "id": relationship.source_ref
                    })
            else:
                # create a new entry
                if not reverse:
                    id_to_related[relationship.source_ref] = [{
                        "relationship": relationship,
                        "id": relationship.target_ref
                    }]
                else:
                    id_to_related[relationship.target_ref] = [{
                        "relationship": relationship,
                        "id": relationship.source_ref
                    }]
    # all objects of relevant type
    if not reverse:
        targets = thesrc.query([
            Filter('type', '=', target_type),
            Filter('revoked', '=', False)
        ])
    else:
        targets = thesrc.query([
            Filter('type', '=', src_type),
            Filter('revoked', '=', False)
        ])

    # build lookup of stixID to stix object
    id_to_target = {}
    for target in targets:
        id_to_target[target.id] = target

    # build final output mappings
    output = {}
    for stix_id in id_to_related:
        value = []
        for related in id_to_related[stix_id]:
            if not related["id"] in id_to_target:
                continue  # targeting a revoked object
            value.append({
                "object": id_to_target[related["id"]],
                "relationship": related["relationship"]
            })
        output[stix_id] = value
    return output

In [ ]:
def techniques_used_by_groups(thesrc):
    """returns group_id => {technique, relationship} for each technique used by the group and each
       technique used by campaigns attributed to the group."""
    # get all techniques used by groups
    techniques_used_by_groups = get_related(thesrc, "intrusion-set", "uses", "attack-pattern") # group_id => {technique, relationship}

    # get groups attributing to campaigns and all techniques used by campaigns
    campaigns_attributed_to_group = {
        "campaigns": get_related(thesrc, "campaign", "attributed-to", "intrusion-set", reverse=True), # group_id => {campaign, relationship}
        "techniques": get_related(thesrc, "campaign", "uses", "attack-pattern") # campaign_id => {technique, relationship}
    }

    for group_id in campaigns_attributed_to_group["campaigns"]:
        techniques_used_by_campaigns = []
        # check if attributed campaign is using technique
        for campaign in campaigns_attributed_to_group["campaigns"][group_id]:
            campaign_id = campaign["object"]["id"]
            if campaign_id in campaigns_attributed_to_group["techniques"]:
                techniques_used_by_campaigns.extend(campaigns_attributed_to_group["techniques"][campaign_id])

        # update techniques used by groups to include techniques used by a groups attributed campaign
        if group_id in techniques_used_by_groups:
            techniques_used_by_groups[group_id].extend(techniques_used_by_campaigns)
        else:
            techniques_used_by_groups[group_id] = techniques_used_by_campaigns
    return techniques_used_by_groups

In [ ]:
techniques_used_by_groups_obj = techniques_used_by_groups(mitre_matrix) 
# intrusion_sets_list = list(techniques_used_by_groups_obj.keys())# Guardamos en una lista cada una de las intrusion-sets almacenadas
techniques_used_by_groups_obj

KeyError: 0

In [ ]:
result_query = mitre_matrix.query([Filter('external_references.external_id', '=', 'T1055.011')])
result_query[0]['x_mitre_deprecated']

KeyError: 'x_mitre_deprecated'

In [ ]:
techniques_df.to_csv(r'C:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\stix2\test_nofilter.csv', sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)